# Import

In [ ]:
import numpy as np
import pandas as pd
from math import sqrt
import matplotlib.pyplot as plt
from matplotlib import rcParams
from keras.models import Sequential
from keras.layers import Dense
from keras.layers import LSTM
from keras.layers import Flatten
from keras.layers import Conv2D
from keras.layers import MaxPooling2D
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_absolute_percentage_error

%matplotlib inline
#%tensorflow_version 1.x
import tensorflow as tf
print(tf.__version__)


## load dataset

In [ ]:
# 1) Load & merge two stations 
stations = ['eMalahleni','Middelburg']

In [ ]:
# 1) Load & merge two stations 
stations = ['eMalahleni','Middelburg']
dfs = []
for st in stations:
    df = pd.read_csv(f'C:\\Users\\User\\Documents\\GitHub\\Health-impacts-of-air-pollution\\AirData\\{st}IM.csv', sep=';', header=0, index_col=0)
    # rename to keep track
    df.columns = [f'{st}_{c}' for c in df.columns]
    dfs.append(df)
data = pd.concat(dfs, axis=1).dropna()

values = data.values

## Plot pm2.5

In [ ]:
plt.plot(values[:,0])
plt.ylabel(data.columns[0])
plt.show()

## Data preparation

We need a way to prepare the data for whatever way we would like to formulate the problem.

In this case we are formulating it such that we take in 1 time step input (14 variables) and output 1 time step output (1 variable). In other words we are trying to solve the following question: given the pollution and weather conditions of the previous hour, can we predict the PM2.5 level for the next hour.

The single variable we are outputing is the PM2.5 level. Note we also use PM2.5 level in our input.

Credit for this code: https://machinelearningmastery.com/convert-time-series-supervised-learning-problem-python/

In [ ]:
def series_to_supervised(data, n_in=1, n_out=1, dropnan=True):
    n_vars = 1 if type(data) is list else data.shape[1]
    df = pd.DataFrame(data)
    cols, names = list(), list()
    # input sequence (t-n, ... t-1)
    for i in range(n_in, 0, -1):
        cols.append(df.shift(i))
        names += [('var%d(t-%d)' % (j+1, i)) for j in range(n_vars)]
    # forecast sequence (t, t+1, ... t+n)
    for i in range(0, n_out):
        cols.append(df.shift(-i))
        if i == 0:
            names += [('var%d(t)' % (j+1)) for j in range(n_vars)]
        else:
            names += [('var%d(t+%d)' % (j+1, i)) for j in range(n_vars)]
    # put it all together
    agg = pd.concat(cols, axis=1)
    agg.columns = names
    # drop rows with NaN values
    if dropnan:
        agg.dropna(inplace=True)
    return agg

## Get column names

In [ ]:
data.columns

##Actually perform the data preparation

We scale the values between 0 and 1.

The code which converts the data into the suitable way we want, in this case, will produce 14 output variables. In our case we only want to predict PM2.5, that is why we drop the other collumns from the dataframe.

Credit for this code: https://machinelearningmastery.com/multivariate-time-series-forecasting-lstms-keras/

In [ ]:
# ensure all data is float
values = values.astype('float32')

# normalize features
scaler = MinMaxScaler(feature_range=(0, 1))
scaled = scaler.fit_transform(values)

In [ ]:
# frame as supervised learning
reframed = series_to_supervised(scaled, 1, 1)

n_vars = scaled.shape[1] 

drop = list(range(n_vars+1, 2*n_vars))
reframed.drop(reframed.columns[drop], axis=1, inplace=True)
values = reframed.values

## View the data

In [ ]:
reframed.head()

## Create X and Y variables

In [ ]:
values.shape

In [ ]:
X = values[:,:-1]

In [ ]:
Y = values[:,-1]

## Check the shapes

In [ ]:
X.shape

In [ ]:
Y.shape

## Reshaping

reshape from [samples, timesteps] into [samples, timesteps, stations, features]

In [ ]:
n_stations = len(stations)  
n_feats    = int(n_vars / n_stations)

In [ ]:
X = X.reshape(X.shape[0],n_stations,n_feats,1)

In [ ]:
X.shape

## Training, validation and testing split

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.20, random_state=42)
X_train, X_val, Y_train, Y_val = train_test_split(X_train, Y_train, test_size=0.20, random_state=42)

## Check the shape

In [ ]:
print ('X_train:',X_train.shape)
print ('Y_train:',Y_train.shape)
print ()
print ('X_val:',X_val.shape)
print ('Y_val:',Y_val.shape)
print ()
print ('X_test:',X_test.shape)
print ('Y_test:',Y_test.shape)

## Define a model
Credit for this code: https://machinelearningmastery.com/how-to-develop-lstm-models-for-time-series-forecasting/

In [ ]:
model = Sequential()

# 1) 2D convolution over your 2×13 grid
model.add(Conv2D(filters=128,kernel_size=(2,2),      # e.g. covers 2 stations × 2 features
    activation='relu', padding='same',input_shape=(n_stations, n_feats, 1)))
# downsample if you like:
model.add(MaxPooling2D(pool_size=(1,2)))  # halves the feature dimension
# you can stack more Conv2D/Pool2D blocks:
model.add(Conv2D(128, (2,2), activation='relu', padding='same'))
model.add(MaxPooling2D((1,2)))
model.add(Conv2D(128, (2,2), activation='relu', padding='same'))
model.add(MaxPooling2D((1,2)))
# flatten to go into dense head
model.add(Flatten())
model.add(Dense(64, activation='relu'))
model.add(Dense(1, activation='linear'))  # for regression

model.compile(optimizer='adam', loss='mse')

## Print summary

In [ ]:
model.summary()

## Training

In [ ]:
history = model.fit(X_train, Y_train, validation_data=(X_val, Y_val), epochs=50, batch_size=16, verbose=1)

## Predict

In [ ]:
prediction = model.predict(X_test)

In [ ]:
def unscale(scaled_value):
    # if target variable is the first column, then, data_max_[0]
    unscaled_value = scaled_value * (scaler.data_max_[0] - scaler.data_min_[0]) + (scaler.data_min_[0])
    return unscaled_value

In [ ]:
predictions = unscale(prediction)

In [ ]:
Y_tests = unscale(Y_test)

In [ ]:
mae = mean_absolute_error(Y_tests, predictions)

In [ ]:
rmse = sqrt(mean_squared_error(Y_tests, predictions))

In [ ]:
r2 = r2_score(Y_tests, predictions)

In [ ]:
print(f"MAE:  {mae:.6f}")
print(f"RMSE: {rmse:.6f}")
print(f"R²:   {r2:.6f}")

# Plot the performance

## Compare prediction and testing data

In [ ]:
rcParams['font.weight'] = 'bold'
plt.plot(Y_tests[0:240], color='blue', label = 'Observed')
plt.plot(predictions[0:240], color='red', label = 'Predicted')
plt.ylabel('PM2.5', fontname="Times New Roman", size=20,fontweight="bold")
plt.xlabel('Time(Hrs)', fontname="Times New Roman", size=20,fontweight="bold")
plt.title('eMalahleni CNN Model', fontname="Times New Roman", size=28,fontweight="bold")
legend_properties = {'weight':'bold'}
plt.legend(prop=legend_properties)
#plt.savefig("C:/Users/User/Documents/GitHub/Health-impacts-of-air-pollution/PLOTS/eMCNNPred.png")  
plt.savefig("C:/Users/User/Documents/GitHub/Health-impacts-of-air-pollution/PLOTS/2DCNN/eMa2DCNNPred_PM2.png", dpi=300, bbox_inches='tight')
plt.show()

# Test

In [ ]:
## Calculate errors
errors = predictions.flatten() - Y_tests

# Calculate quantiles based on actual values
quantiles, bins = pd.qcut(Y_tests, q=10, duplicates='drop', retbins=True)

# Calculate average error for each quantile
quantile_errors = []
for i in range(len(bins) - 1):
    group_indices = np.where((Y_tests >= bins[i]) & (Y_tests < bins[i+1]))[0]
    quantile_errors.append(errors[group_indices].mean())

# Plot quantiles vs. average errors
rcParams['font.weight'] = 'bold'
plt.figure(figsize=(8, 6))
plt.plot(range(1, len(quantiles.categories) + 1), quantile_errors, marker='o')
plt.xlabel('Quantile', fontname="Times New Roman", size=20,fontweight="bold")
plt.ylabel('Average Error', fontname="Times New Roman", size=20,fontweight="bold")
plt.title('eMalahleni CNN Model', fontname="Times New Roman", size=28,fontweight="bold")
plt.xticks(range(1, len(quantiles.categories) + 1), [str(q) for q in quantiles.categories], rotation=45)
plt.grid(True)
plt.savefig("C:/Users/User/Documents/GitHub/Health-impacts-of-air-pollution/PLOTS/2DCNN/eMa2DCNNQuan_PM2.png", dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
errors = predictions.flatten() - Y_tests

# Calculate quantiles based on actual values
quantiles, bins = pd.qcut(Y_tests, q=10, duplicates='drop', retbins=True)

# Calculate average error for each quantile
quantile_errors = []
for i in range(len(bins) - 1):
    group_indices = np.where((Y_tests >= bins[i]) & (Y_tests < bins[i+1]))[0]
    quantile_errors.append(errors[group_indices].mean())

# Round the bin edges for better readability
rounded_bins = np.round(bins, decimals=3)

# Plot quantiles vs. average errors
rcParams['font.weight'] = 'bold'
plt.figure(figsize=(8, 6))
plt.plot(range(1, len(quantiles.categories) + 1), quantile_errors, marker='o')
plt.xlabel('Quantile', fontname="Times New Roman", size=20, fontweight="bold")
plt.ylabel('Average Error', fontname="Times New Roman", size=20, fontweight="bold")
plt.title('eMalahleni CNN Model', fontname="Times New Roman", size=28, fontweight="bold")
plt.xticks(range(1, len(quantiles.categories) + 1), [f'{rounded_bins[i]:.3f} - {rounded_bins[i+1]:.3f}' for i in range(len(rounded_bins) - 1)], rotation=45)
plt.grid(True)
plt.savefig("C:/Users/User/Documents/GitHub/Health-impacts-of-air-pollution/PLOTS/2DCNN/eMa2DCNNQuan_PM2.png", dpi=300, bbox_inches='tight')
plt.show()

# SHAP

In [ ]:
# ─── PART 3: SHAP EXPLAINER ───
import shap

B = min(300, X_train.shape[0])
bg_idx   = np.random.choice(X_train.shape[0], B, replace=False)
bg_flat  = X_train[bg_idx].reshape(B, -1)

# test_flat for all or a subset of test samples
T = min(60, X_test.shape[0])
test_idx   = np.random.choice(X_test.shape[0], T, replace=False)
test_flat  = X_test[test_idx].reshape(T, -1)

# 2) wrap your model.predict so it accepts 2D input
f = lambda x: model.predict(
    x.reshape(-1, n_stations, n_feats, 1)
).flatten()

# 3) build the explainer and compute SHAP values
explainer   = shap.KernelExplainer(f, bg_flat)
shap_vals   = explainer.shap_values(test_flat)

# 4) feature names = your original column names
feat_names = list(data.columns)  # length = n_stations * n_feats

# 5) summary plot
shap.summary_plot(
    shap_vals,
    test_flat,
    feature_names=feat_names,
    show=False)
plt.title('eMalahleni CNN Model', fontweight='bold', size=14)

plt.savefig("C:/Users/User/Documents/GitHub/Health-impacts-of-air-pollution/PLOTS/2DCNN/eMa2DCNNshap_PM2.png", dpi=300, bbox_inches='tight')


# Save Model

In [ ]:
# eMalahleni PM2.5
model.save('2Dcnn_model.h5')
#model.save('2Dcnn_modelPM1.h5')
#model.save('2Dcnn_modelSO2.h5')
#model.save('2Dcnn_modelNO2.h5')

In [ ]:
# Ermelo PM2.5
model.save('2Dcnn_modelEPM2.h5')
#model.save('2Dcnn_modelEPM1.h5')
#model.save('2Dcnn_modelESO2.h5')
#model.save('2Dcnn_modelENO2.h5')

In [ ]:
# Hendrina PM2.5
#model.save('2Dcnn_modelHPM2.h5')
#model.save('2Dcnn_modelHPM1.h5')
#model.save('2Dcnn_modelHSO2.h5')
#model.save('2Dcnn_modelHNO2.h5')

In [ ]:
# Middelburg PM2.5
model.save('2Dcnn_modelMPM2.h5')
#model.save('2Dcnn_modelMPM1.h5')
#model.save('2Dcnn_modelMSO2.h5')
#model.save('2Dcnn_modelMNO2.h5')

In [ ]:
# Secunda PM2.5
model.save('2Dcnn_modelSPM2.h5')
#model.save('2Dcnn_modelSPM1.h5')
#model.save('2Dcnn_modelSSO2.h5')
#model.save('2Dcnn_modelSNO2.h5')

# SAVE THE RESULTS

In [ ]:
results_df = pd.DataFrame({
    'actual': Y_tests.ravel(),
    'predicted': predictions.ravel(),
    'error': errors.ravel()
})

results_df.to_csv("C:/Users/User/Documents/GitHub/Health-impacts-of-air-pollution/PLOTS/2DCNN/eMa2DCNN_predictions_PM2.csv", index=False)

In [ ]:
# Save metrics
metrics_dict = {
    'MAE': mae,
    'RMSE': rmse,
    'R2': r2
}

metrics_df = pd.DataFrame([metrics_dict])
metrics_df.to_csv("C:/Users/User/Documents/GitHub/Health-impacts-of-air-pollution/PLOTS/2DCNN/eMa2DCNN_metrics_PM2.csv", index=False)

# Forecast

In [ ]:
mean_absolute_error(Y_test[1:6], prediction[1:6])

In [ ]:
mean_absolute_error(Y_test[1:12], prediction[1:12])

In [ ]:
mean_absolute_error(Y_test[1:18], prediction[1:18])

In [ ]:
mean_absolute_error(Y_test[1:24], prediction[1:24])

In [ ]:
mean_absolute_error(Y_test[1:36], prediction[1:36])

In [ ]:
mean_absolute_error(Y_test[1:48], prediction[1:48])

In [ ]:
rmse = sqrt(mean_squared_error(Y_test[1:6], prediction[1:6]))
print(rmse)

In [ ]:
rmse = sqrt(mean_squared_error(Y_test[1:12], prediction[1:12]))
print(rmse)

In [ ]:
rmse = sqrt(mean_squared_error(Y_test[1:18], prediction[1:18]))
print(rmse)

In [ ]:
rmse = sqrt(mean_squared_error(Y_test[1:24], prediction[1:24]))
print(rmse)

In [ ]:
rmse = sqrt(mean_squared_error(Y_test[1:36], prediction[1:36]))
print(rmse)

In [ ]:
rmse = sqrt(mean_squared_error(Y_test[1:48], prediction[1:48]))
print(rmse)